# 06. Prep for Surrogate Modeling

This notebook prepares the dataset for the **Neuro-Fuzzy–inspired adaptive reasoning model**.
It constructs the input state vector (Archetype Percentages, Deltas) and generates a **heuristic proxy target** (Adaptation Multiplier) to shape the model's reasoning surface.

## 1. Setup and Imports

In [1]:
import pandas as pd
import os

# Inputs & Outputs
INPUT_PATH = 'data/processed/5_clustered_telemetry.csv'
OUTPUT_PATH = 'data/processed/6_anfis_dataset.csv'

print("Libraries imported.")

Libraries imported.


## 2. Load Data

In [2]:
print("Loading clustered telemetry data...")
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(df)} rows.")
else:
    raise FileNotFoundError(f"File not found at {INPUT_PATH}")

Loading clustered telemetry data...
Loaded 2838 rows.


## 3. Define Input Features
We select the state variables (Archetype Percentages) and change variables (Deltas) that will feed into the ANFIS model.

In [3]:
# Select Inputs for ANFIS
# 1. State: Archetype Percentages (Combat, Collect, Explore)
# 2. Change: Deltas (Delta_Combat, Delta_Collect, Delta_Explore)
input_features = ['pct_combat', 'pct_collect', 'pct_explore', 
                  'delta_combat', 'delta_collect', 'delta_explore']

print("Input features selected:")
print(input_features)

Input features selected:
['pct_combat', 'pct_collect', 'pct_explore', 'delta_combat', 'delta_collect', 'delta_explore']


## 4. Generate Heuristic Proxy Target (Adaptation Multiplier)
The target multiplier generates a difficulty adjustment proxy in replace of explicit designer labels.

**Logic**:
- **Base Multiplier**: 1.0
- **Penalty (Easier)**: -0.1 per death
- **Reward (Harder)**: +0.05 * Normalized Total Activity Score (Rewards high engagement)

In [4]:
# Normalize Total Activity Score for Intensity Calculation
if 'score_total' in df.columns:
    max_score = df['score_total'].max()
    if max_score > 0:
        df['activity_intensity'] = df['score_total'] / max_score
    else:
        df['activity_intensity'] = 0.0
    print(f"Activity Intensity normalized. Max Score: {max_score}")
else:
    print("Warning: 'score_total' not found. Using default intensity 0.5")
    df['activity_intensity'] = 0.5

Activity Intensity normalized. Max Score: 16460.265218571312


In [5]:
# Apply Heuristic Formula
print("Calculating target multiplier...")
df['target_multiplier'] = 1.0 - (0.1 * df['death_count'].fillna(0)) + (0.05 * df['activity_intensity'])

# Clip to [0.5, 1.5] for stability
df['target_multiplier'] = df['target_multiplier'].clip(0.5, 1.5)

print("Target multiplier calculated and clipped to [0.5, 1.5].")

Calculating target multiplier...
Target multiplier calculated and clipped to [0.5, 1.5].


## 5. Export ANFIS Dataset
We save the final dataset containing User IDs, Timestamps, Inputs, and the Target.

In [6]:
# Save Dataset
final_cols = ['userId', 'timestamp', 'cluster'] + input_features + ['target_multiplier']
anfis_df = df[final_cols].copy()

anfis_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved ANFIS dataset to: {OUTPUT_PATH}")
print("\n--- Target Stats ---")
print(anfis_df['target_multiplier'].describe())

Saved ANFIS dataset to: data/processed/6_anfis_dataset.csv

--- Target Stats ---
count    2838.000000
mean        1.024050
std         0.011678
min         1.000093
25%         1.016122
50%         1.024298
75%         1.032459
max         1.050000
Name: target_multiplier, dtype: float64
